<a href="https://colab.research.google.com/github/Crmorale/Cpp_modules/blob/main/Criptograf%C3%ADa_aplicada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 📦 Instala la librería si no está instalada (solo necesario en Colab o entornos nuevos)
!pip install cryptography ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.7 MB/s eta 0:00:00


# Prueba de esfuerzo

Este código busca el número N que hacer que el hash de mensaje y N concatenado tenga los primeros k bits a cero.

In [ ]:
import hashlib
import time

def prueba_de_esfuerzo(mensaje, k):
    """
    Encuentra un número N tal que el hash SHA-256 de (mensaje + str(N))
    comience con al menos k bits en cero.

    Parámetros:
    - mensaje: texto base (string)
    - k: cantidad de ceros iniciales requeridos en el hash (en bits)

    Retorna:
    - N encontrado
    - hash correspondiente
    - tiempo usado
    """
    start_time = time.time()
    N = 0
    objetivo = '0' * k  # k ceros en binario

    while True:
        combinado = mensaje + str(N)
        hash_binario = bin(int(hashlib.sha256(combinado.encode()).hexdigest(), 16))[2:].zfill(256)
        if hash_binario.startswith(objetivo):
            break
        N += 1

    tiempo = time.time() - start_time
    hash_hex = hashlib.sha256((mensaje + str(N)).encode()).hexdigest()

    return N, hash_hex, tiempo

# === 🔧 Parámetros de entrada ===
mensaje = "Hola mundo2"
k = 19  # Cambia esto para ajustar la dificultad

# === 🚀 Ejecución ===
N, hash_encontrado, tiempo = prueba_de_esfuerzo(mensaje, k)

print(f"Mensaje: {mensaje}")
print(f"Bits en cero requeridos: {k}")
print(f"Nonce encontrado: {N}")
print(f"Hash resultante: {hash_encontrado}")
print(f"Tiempo de búsqueda: {tiempo:.2f} segundos")

Mensaje: Hola mundo2
Bits en cero requeridos: 19
Nonce encontrado: 9381
Hash resultante: 000011ea7f6f8b1f5bdd64f0bbdd62149e3b9300ede59391b87876367d454374
Tiempo de búsqueda: 0.01 segundos


El siguiente bloque de código incluye una pequeña modificación para que se pueda indicar por donde empezar a buscar N. Podéis trabajar por parejas o en equipos de más personas para acelerar la busqueda de N, repartiendo el espacio de busqueda, de forma que cada miembro del equipo empiece en un lugar diferente.


In [ ]:
import hashlib
import time

def prueba_de_esfuerzo(mensaje, k, start):
    """
    Encuentra un número N tal que el hash SHA-256 de (mensaje + str(N))
    comience con al menos k bits en cero.

    Parámetros:
    - mensaje: texto base (string)
    - k: cantidad de ceros iniciales requeridos en el hash (en bits)

    Retorna:
    - N encontrado
    - hash correspondiente
    - tiempo usado
    """
    start_time = time.time()
    N = start
    objetivo = '0' * k  # k ceros en binario

    while True:
        combinado = mensaje + str(N)
        hash_binario = bin(int(hashlib.sha256(combinado.encode()).hexdigest(), 16))[2:].zfill(256)
        if hash_binario.startswith(objetivo):
            break
        N += 1

    tiempo = time.time() - start_time
    hash_hex = hashlib.sha256((mensaje + str(N)).encode()).hexdigest()

    return N, hash_hex, tiempo

# === 🔧 Parámetros de entrada ===
mensaje = "Hola mundo"
k = 23  # Cambia esto para ajustar la dificultad

# === 🚀 Ejecución ===
N, hash_encontrado, tiempo = prueba_de_esfuerzo(mensaje, k, 15633768)

print(f"Mensaje: {mensaje}")
print(f"Bits en cero requeridos: {k}")
print(f"Nonce encontrado: {N}")
print(f"Hash resultante: {hash_encontrado}")
print(f"Tiempo de búsqueda: {tiempo:.2f} segundos")

Mensaje: Hola mundo
Bits en cero requeridos: 23
Nonce encontrado: 16633768
Hash resultante: 000001a8355950495ce0406680d927d13fe091c60d5280a0130178e632b8e748
Tiempo de búsqueda: 2.00 segundos


# Lanzamiento de Dados

Este código busca mostrar como funciona el protocolo de lanzamiento de dados. Alice envía a Bob 6 mensajes (caras de un dado)  con su clave pública, Bob elige uno al azar y se lo devuelve, Alice lo descifra y revela el resultado, y finalmente Alice envía sus claves a Bob para verificación.

## Generación de claves

Alice genera un par de claves (pública y privada) utilizando la librería `cryptography`.


In [ ]:
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.backends import default_backend

# === Generación de claves de Alice ===

def generar_claves_alice():
    """
    Genera un par de claves RSA (pública y privada) para Alice.

    Retorna:
    - private_key_alice: La clave privada de Alice.
    - public_key_alice: La clave pública de Alice.
    """
    clave_privada_alice = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
        backend=default_backend()
    )
    clave_publica_alice = clave_privada_alice.public_key()
    return clave_privada_alice, clave_publica_alice

# Generar las claves al ejecutar la celda
clave_privada_alice, clave_publica_alice = generar_claves_alice()

## Cifrado de mensajes

Alice cifra 6 mensajes, cada uno representando una cara del dado (del 1 al 6), usando su clave pública.


In [ ]:
from cryptography.hazmat.primitives import hashes
import random

# === Cifrado de mensajes por Alice ===

def cifrar_caras_dado(clave_publica):
    """
    Cifra los mensajes que representan las caras de un dado (1 al 6)
    usando la clave pública proporcionada.

    Parámetros:
    - clave_publica: La clave pública a utilizar para el cifrado.

    Retorna:
    - mensajes_cifrados: Una lista de mensajes cifrados (en bytes).
    """
    caras_dado = ["1", "2", "3", "4", "5", "6"]
    random.shuffle(caras_dado)
    mensajes_cifrados = []

    for cara in caras_dado:
        cara_codificada = cara.encode()
        mensaje_cifrado = clave_publica.encrypt(
            cara_codificada,
            padding.OAEP(
                mgf=padding.MGF1(algorithm=hashes.SHA256()),
                algorithm=hashes.SHA256(),
                label=None
            )
        )
        mensajes_cifrados.append(mensaje_cifrado)

    return mensajes_cifrados

# Cifrar las caras del dado al ejecutar la celda
mensajes_cifrados = cifrar_caras_dado(clave_publica_alice)

## Selección de mensaje

Bob recibe los mensajes cifrados, elige uno al azar y se lo envía de vuelta a Alice.


In [ ]:
import random

# === Selección de mensaje por Bob ===

def seleccionar_mensaje_cifrado(mensajes_cifrados):
    """
    Bob selecciona uno de los mensajes cifrados al azar.

    Parámetros:
    - mensajes_cifrados: Una lista de mensajes cifrados.

    Retorna:
    - mensaje_cifrado_seleccionado: El mensaje cifrado elegido por Bob.
    """
    mensaje_cifrado_seleccionado = random.choice(mensajes_cifrados)
    return mensaje_cifrado_seleccionado

# Bob selecciona un mensaje al ejecutar la celda
mensaje_cifrado_seleccionado = seleccionar_mensaje_cifrado(mensajes_cifrados)

## Descifrado y revelación

Alice descifra el mensaje elegido por Bob usando su clave privada y le revela el resultado (el número del dado).


In [ ]:
# === Descifrado y revelación por Alice ===

def descifrar_mensaje(clave_privada, mensaje_cifrado):
    """
    Alice descifra un mensaje cifrado usando su clave privada.

    Parámetros:
    - clave_privada: La clave privada de Alice.
    - mensaje_cifrado: El mensaje cifrado a descifrar.

    Retorna:
    - resultado_revelado: El mensaje descifrado como string.
    """
    mensaje_descifrado_bytes = clave_privada.decrypt(
        mensaje_cifrado,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    resultado_revelado = mensaje_descifrado_bytes.decode()
    return resultado_revelado

# Alice descifra el mensaje al ejecutar la celda
resultado_revelado = descifrar_mensaje(clave_privada_alice, mensaje_cifrado_seleccionado)

# Imprimir el resultado revelado
print(f"Alice descifró el mensaje de Bob y reveló el resultado: {resultado_revelado}")

Alice descifró el mensaje de Bob y reveló el resultado: 3


## Verificación

Alice envía a Bob su clave pública y privada para que Bob pueda verificar que los mensajes originales correspondían a los números del 1 al 6 y que el resultado revelado es correcto.


In [ ]:
# === Verificación por Bob ===

def verificar_protocolo(clave_publica, clave_privada, mensajes_cifrados_originales, mensaje_cifrado_seleccionado, resultado_revelado):
    """
    Bob verifica que los mensajes originales correspondían a las caras del dado
    y que el resultado revelado es correcto usando las claves de Alice.

    Parámetros:
    - clave_publica: La clave pública de Alice.
    - clave_privada: La clave privada de Alice.
    - mensajes_cifrados_originales: La lista original de mensajes cifrados por Alice.
    - mensaje_cifrado_seleccionado: El mensaje cifrado que Bob seleccionó.
    - resultado_revelado: El resultado que Alice reveló.
    """
    print("\n=== Paso de Verificación por Bob ===")
    print("Bob ha recibido las claves de Alice para verificar el protocolo.")


    # 1. Bob usa la clave pública para cifrar 1-6 y compara con los mensajes originales cifrados
    print("\nBob verifica los mensajes cifrados originales cifrando 1-6 con la clave pública de Alice:")
    mensajes_cifrados_verificacion_bob = []
    for cara in range(1, 7):
        cara_codificada = str(cara).encode()
        mensaje_cifrado_bob = clave_publica.encrypt(
            cara_codificada,
            padding.OAEP(
                mgf=padding.MGF1(algorithm=hashes.SHA256()),
                algorithm=hashes.SHA256(),
                label=None
            )
        )
        mensajes_cifrados_verificacion_bob.append(mensaje_cifrado_bob)

    # Comparar los mensajes cifrados de verificación de Bob con los mensajes originales de Alice
    # Nota: Debido al padding OAEP, la comparación directa de los textos cifrados no es posible
    # de forma determinista. Esto es intencional y crucial en RSA para prevenir ataques de diccionario.
    # Aunque Bob conozca la clave pública de Alice y pueda cifrar los valores 1-6,
    # no puede determinar cuál mensaje cifrado original corresponde a cada número
    # simplemente comparando los textos cifrados.
    print("Comparación de los mensajes cifrados de verificación de Bob con los mensajes originales de Alice:")
    if sorted(mensajes_cifrados_verificacion_bob) == sorted(mensajes_cifrados_originales):
         print("  Verificación exitosa: Los mensajes cifrados de Bob coinciden con los mensajes originales de Alice.")
    else:
         print("  Verificación fallida: Los mensajes cifrados de Bob NO coinciden con los mensajes originales de Alice (esperado y deseado debido al padding OAEP). Esto impide que Bob adivine los valores ocultos.")


    # 2. Bob verifica el resultado revelado descifrando el mensaje seleccionado con la clave privada
    print("\nBob verifica el resultado revelado utilizando la clave privada proporcionada:")
    try:
        resultado_descifrado_bob = clave_privada.decrypt(
            mensaje_cifrado_seleccionado,
            padding.OAEP(
                mgf=padding.MGF1(algorithm=hashes.SHA256()),
                algorithm=hashes.SHA256(),
                label=None
            )
        ).decode()

        if resultado_descifrado_bob == resultado_revelado:
            print(f"  Verificación exitosa: El descifrado de Bob del mensaje seleccionado coincide con el resultado revelado ('{resultado_revelado}').")
        else:
            print(f"  Verificación fallida: El descifrado de Bob ('{resultado_descifrado_bob}') NO coincide con el resultado revelado ('{resultado_revelado}').")
    except Exception as e:
        print(f"  Verificación fallida durante el descifrado: {e}")

    # 3. Bob descifra todos los mensajes iniciales con la clave privada para verificar su contenido
    print("\nBob descifra todos los mensajes iniciales con la clave privada de Alice para verificar su contenido:")
    resultados_descifrados_iniciales = []
    for i, msg in enumerate(mensajes_cifrados_originales):
        try:
            resultado_descifrado = clave_privada.decrypt(
                msg,
                padding.OAEP(
                    mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(),
                    label=None
                )
            ).decode()
            resultados_descifrados_iniciales.append(resultado_descifrado)
            print(f"  Mensaje {i+1} descifrado: {resultado_descifrado}")
        except Exception as e:
            print(f"  Mensaje {i+1} no se pudo descifrar: {e}")

    # Verificar si los resultados descifrados son las caras del dado 1-6
    caras_esperadas = [str(i) for i in range(1, 7)]
    if sorted(resultados_descifrados_iniciales) == sorted(caras_esperadas):
        print("  Verificación exitosa: Los mensajes iniciales descifrados corresponden a las caras del dado (1-6).")
    else:
        print("  Verificación fallida: Los mensajes iniciales descifrados NO corresponden a las caras del dado (1-6).")



# Ejecutar la verificación al ejecutar la celda
verificar_protocolo(clave_publica_alice, clave_privada_alice, mensajes_cifrados, mensaje_cifrado_seleccionado, resultado_revelado)


=== Paso de Verificación por Bob ===
Bob ha recibido las claves de Alice para verificar el protocolo.

Bob verifica los mensajes cifrados originales cifrando 1-6 con la clave pública de Alice:
Comparación de los mensajes cifrados de verificación de Bob con los mensajes originales de Alice:
  Verificación fallida: Los mensajes cifrados de Bob NO coinciden con los mensajes originales de Alice (esperado y deseado debido al padding OAEP). Esto impide que Bob adivine los valores ocultos.

Bob verifica el resultado revelado utilizando la clave privada proporcionada:
  Verificación exitosa: El descifrado de Bob del mensaje seleccionado coincide con el resultado revelado ('3').

Bob descifra todos los mensajes iniciales con la clave privada de Alice para verificar su contenido:
  Mensaje 1 descifrado: 4
  Mensaje 2 descifrado: 1
  Mensaje 3 descifrado: 6
  Mensaje 4 descifrado: 3
  Mensaje 5 descifrado: 5
  Mensaje 6 descifrado: 2
  Verificación exitosa: Los mensajes iniciales descifrados cor

## Simulación paso a paso

La simulación demuestra con éxito un esquema de compromiso básico donde Alice se compromete con un conjunto de resultados (las caras del dado) sin revelarlos inicialmente, Bob selecciona uno, y Alice luego revela el resultado elegido y proporciona prueba (sus claves) de que el resultado fue parte del conjunto original.



In [ ]:
import hashlib
import random
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.backends import default_backend

# === Ejecución del Protocolo ===
print("=== Inicio del Protocolo de Lanzamiento de Dados ===")

# Paso 1: Generación de claves de Alice
clave_privada_alice, clave_publica_alice = generar_claves_alice()
print("Paso 1 completado: Claves de Alice generadas.")

# Paso 2: Cifrado de mensajes por Alice
mensajes_cifrados = cifrar_caras_dado(clave_publica_alice)
print(f"Paso 2 completado: Alice ha cifrado {len(mensajes_cifrados)} mensajes.")

# Paso 3: Selección de mensaje por Bob
mensaje_cifrado_seleccionado = seleccionar_mensaje_cifrado(mensajes_cifrados)
print("Paso 3 completado: Bob ha seleccionado un mensaje cifrado al azar.")

# Paso 4: Descifrado y revelación por Alice
resultado_revelado = descifrar_mensaje(clave_privada_alice, mensaje_cifrado_seleccionado)
print(f"Paso 4 completado: Alice descifró el mensaje de Bob y reveló el resultado: {resultado_revelado}")

# Paso 5: Verificación por Bob
verificar_protocolo(clave_publica_alice, clave_privada_alice, mensajes_cifrados, mensaje_cifrado_seleccionado, resultado_revelado)
print("Paso 5 completado: Verificación por Bob realizada.")

print("\n=== Protocolo de Lanzamiento de Dados Finalizado ===")

# --- Optional: Include the prueba_de_esfuerzo function call from earlier ---
# print("\n--- Prueba de Esfuerzo (from earlier notebook state) ---")
# mensaje_poe = "Hola mundo"
# k_poe = 23
# start_poe = 15633768
# N_poe, hash_encontrado_poe, tiempo_poe = prueba_de_esfuerzo(mensaje_poe, k_poe, start_poe)
# print(f"Mensaje: {mensaje_poe}")
# print(f"Bits en cero requeridos: {k_poe}")
# print(f"Nonce encontrado: {N_poe}")
# print(f"Hash resultante: {hash_encontrado_poe}")
# print(f"Tiempo de búsqueda: {tiempo_poe:.2f} segundos")

=== Inicio del Protocolo de Lanzamiento de Dados ===
Paso 1 completado: Claves de Alice generadas.
Paso 2 completado: Alice ha cifrado 6 mensajes.
Paso 3 completado: Bob ha seleccionado un mensaje cifrado al azar.
Paso 4 completado: Alice descifró el mensaje de Bob y reveló el resultado: 3

=== Paso de Verificación por Bob ===
Bob ha recibido las claves de Alice para verificar el protocolo.

Bob verifica los mensajes cifrados originales cifrando 1-6 con la clave pública de Alice:
Comparación de los mensajes cifrados de verificación de Bob con los mensajes originales de Alice:
  Verificación fallida: Los mensajes cifrados de Bob NO coinciden con los mensajes originales de Alice (esperado y deseado debido al padding OAEP). Esto impide que Bob adivine los valores ocultos.

Bob verifica el resultado revelado utilizando la clave privada proporcionada:
  Verificación exitosa: El descifrado de Bob del mensaje seleccionado coincide con el resultado revelado ('3').

Bob descifra todos los mensa

# Protocolo de Compromiso

Un **protocolo de compromiso** es un protocolo criptográfico que permite a una parte (Alice) comprometerse con un valor o un mensaje, pero sin revelarlo inicialmente a la otra parte (Bob). En una fase posterior, Alice puede revelar el valor, y Bob puede verificar que el valor revelado es de hecho aquel con el que se realizó el compromiso originalmente.

Este protocolo se divide en dos fases principales:

1.  **Fase de compromiso (Commitment Phase):** Alice toma el valor secreto y calcula un "compromiso" (commit). Este compromiso se envía a Bob. El compromiso tiene la propiedad de que no revela ninguna información sobre el valor secreto, pero ata a Alice a ese valor particular. Es decir, la Alice no puede cambiar el valor secreto después de haber enviado el compromiso sin que Bob se dé cuenta.

2.  **Fase de revelación (Reveal Phase):** Alice revela el valor secreto a la receptora. Bob utiliza el valor secreto y el compromiso original para verificar que el valor revelado es el mismo con el que Alice se comprometió.

## Utilidad en Subastas Online

En el contexto de una subasta online, un protocolo de compromiso es extremadamente útil para garantizar la equidad y prevenir trampas. Sin un protocolo de compromiso, un postor podría esperar a ver las pujas de otros postores y luego cambiar su propia puja para superarlas por un margen mínimo. Esto se conoce como "sniping" o "último segundo".

Al usar un protocolo de compromiso, cada postor se compromete con su puja antes de que se revelen las pujas de otros. Una vez que el período de compromiso termina, se pasa a la fase de revelación, donde cada postor revela su puja. La plataforma de subastas (o los otros postores, si el protocolo es público) puede verificar que la puja revelada por cada postor es la misma con la que se comprometieron.

Las ventajas de usar un protocolo de compromiso en subastas online incluyen:

*   **Prevención de trampas:** Impide que los postores cambien sus pujas basándose en las pujas de otros.
*   **Equidad:** Asegura que todos los postores se comprometan con sus pujas en igualdad de condiciones.
*   **Transparencia (en la fase de revelación):** Permite verificar que el proceso se llevó a cabo correctamente.

En resumen, un protocolo de compromiso en una subasta online asegura que un postor no pueda retractarse de su puja o alterarla de manera deshonesta una vez que se ha "sellado" su compromiso.

## Generar clave aleatoria

Cada postor genera una clave aleatoria secreta.


In [ ]:
import os

def generar_clave_aleatoria():
    """
    Genera una clave aleatoria secreta.

    Retorna:
    - str: La clave aleatoria en formato hexadecimal.
    """
    clave_bytes = os.urandom(32) # Generar 32 bytes aleatorios
    return clave_bytes.hex()

# Generar claves de ejemplo para múltiples postores
claves_aleatorias_postores = {
    'postor_1': generar_clave_aleatoria(),
    'postor_2': generar_clave_aleatoria(),
    'postor_3': generar_clave_aleatoria()
}

print("Claves aleatorias generadas para los postores:")
for postor, clave in claves_aleatorias_postores.items():
    print(f"  {postor}: {clave}")

Claves aleatorias generadas para los postores:
  postor_1: 742c381f4ad261154b4afbc380ace654aefc288545831d7923c811b43fb84f9b
  postor_2: 6313d8542b784b8010d4c7b9fcad808d8b1a3fc78d7dd6c922349a2ba6714dce
  postor_3: 31ab98e31789e801bb23b9a69485f8e4793dbdaa93ebe3ceb163d699d7d91e87


## Cifrar la puja

Cada postor cifra su puja usando su clave aleatoria. Este es el "compromiso".


In [ ]:
from cryptography.fernet import Fernet
import os
import base64

def generar_clave_fernet():
    """
    Genera una clave Fernet aleatoria.

    Retorna:
    - bytes: La clave Fernet en formato base64 url-safe.
    """
    return Fernet.generate_key()

def cifrar_puja(puja, clave_fernet):
    """
    Cifra la puja usando una clave Fernet.

    Parámetros:
    - puja: La puja como string.
    - clave_fernet: La clave Fernet en formato base64 url-safe bytes.

    Retorna:
    - bytes: La puja cifrada.
    """
    f = Fernet(clave_fernet)
    puja_bytes = puja.encode()
    puja_cifrada = f.encrypt(puja_bytes)
    return puja_cifrada

# Generar claves Fernet y cifrar pujas para múltiples postores
claves_fernet_postores = {
    'postor_1': generar_clave_fernet(),
    'postor_2': generar_clave_fernet(),
    'postor_3': generar_clave_fernet()
}

pujas_originales = {
    'postor_1': "150",
    'postor_2': "250",
    'postor_3': "200"
}

pujas_cifradas_postores = {}
for postor, puja in pujas_originales.items():
    clave_fernet = claves_fernet_postores[postor]
    puja_cifrada = cifrar_puja(puja, clave_fernet)
    pujas_cifradas_postores[postor] = puja_cifrada

print("Pujas originales y cifradas para los postores:")
for postor, puja_cifrada in pujas_cifradas_postores.items():
    print(f"  {postor}: Puja Original = {pujas_originales[postor]}, Puja Cifrada = {puja_cifrada}")

Pujas originales y cifradas para los postores:
  postor_1: Puja Original = 150, Puja Cifrada = b'gAAAAABpA8QjSvfQXvHVwXjdpYi2lnJRMT5h1UHEnH2gTq2OxwjiDwzcw_PwnrSIXjAwuRk-BVyRcsvrnCaHFCaqniHCDXKz6w=='
  postor_2: Puja Original = 250, Puja Cifrada = b'gAAAAABpA8QjViWrwReOwDrtUGBc-3c4FsI2j5A5wxIvDmkhBCXqAsTCNAA33ecWFGn0MdmSJZfPvHgHBuva97Zop6PqVxwPxA=='
  postor_3: Puja Original = 200, Puja Cifrada = b'gAAAAABpA8QjgvuBpR_zlaeJiGim_wq7nCwFy7Q1QPnlJxNhOpKwC7N8yHN91uz5dyR3lD2ZFWU1gfXvI5U8mzptgagqbTbWOA=='


## Envío del compromiso

Los postores envían sus pujas cifradas a Alice (la casa de subastas).


In [ ]:
# Simular la recepción de las pujas cifradas por parte de Alice
pujas_cifradas_recibidas_por_alice = pujas_cifradas_postores
# En un escenario real, Alice recibiría estas pujas de múltiples postores a través de la red

# Mostrar las pujas cifradas recibidas por Alice
print("Pujas cifradas recibidas por Alice:")
print(pujas_cifradas_recibidas_por_alice)

Pujas cifradas recibidas por Alice:
{'postor_1': b'gAAAAABpA8QjSvfQXvHVwXjdpYi2lnJRMT5h1UHEnH2gTq2OxwjiDwzcw_PwnrSIXjAwuRk-BVyRcsvrnCaHFCaqniHCDXKz6w==', 'postor_2': b'gAAAAABpA8QjViWrwReOwDrtUGBc-3c4FsI2j5A5wxIvDmkhBCXqAsTCNAA33ecWFGn0MdmSJZfPvHgHBuva97Zop6PqVxwPxA==', 'postor_3': b'gAAAAABpA8QjgvuBpR_zlaeJiGim_wq7nCwFy7Q1QPnlJxNhOpKwC7N8yHN91uz5dyR3lD2ZFWU1gfXvI5U8mzptgagqbTbWOA=='}


## Fase de revelación (con cifrado)


Cada postor revela su clave aleatoria y su puja original a Alice. Alice verifica entonces que la puja cifrada recibida, cuando se descifra con la clave revelada, coincide con la puja original revelada.

In [ ]:
from cryptography.fernet import Fernet

def descifrar_puja(puja_cifrada, clave_fernet, puja_original_revelada):
    """
    Descifra una puja cifrada usando una clave Fernet y verifica
    si coincide con la puja original revelada.

    Parámetros:
    - puja_cifrada: La puja cifrada como bytes.
    - clave_fernet: La clave Fernet en formato base64 url-safe bytes.
    - puja_original_revelada: La puja original revelada como string.

    Retorna:
    - bool: True si el descifrado coincide con la puja revelada, False en caso contrario.
    """
    try:
        f = Fernet(clave_fernet)
        puja_descifrada_bytes = f.decrypt(puja_cifrada)
        puja_descifrada_str = puja_descifrada_bytes.decode()
        return puja_descifrada_str == puja_original_revelada
    except Exception as e:
        print(f"Error durante el descifrado: {e}")
        return False

# === Simular la fase de revelación para múltiples postores ===
pujas_reveladas_postores = {
    'postor_1': {"puja_original": pujas_originales['postor_1'], "clave_fernet": claves_fernet_postores['postor_1']},
    'postor_2': {"puja_original": pujas_originales['postor_2'], "clave_fernet": claves_fernet_postores['postor_2']},
    'postor_3': {"puja_original": pujas_originales['postor_3'], "clave_fernet": claves_fernet_postores['postor_3']}
}

pujas_reveladas_verificadas = {}

print("\n=== Fase de Revelación y Verificación (con cifrado) ===")
for postor, info_revelada in pujas_reveladas_postores.items():
    puja_original_revelada_str = info_revelada["puja_original"]
    clave_revelada = info_revelada["clave_fernet"]
    puja_cifrada_recibida = pujas_cifradas_recibidas_por_alice.get(postor)

    if puja_cifrada_recibida:
        verificacion_exitosa = descifrar_puja(
            puja_cifrada_recibida,
            clave_revelada,
            puja_original_revelada_str
        )

        if verificacion_exitosa:
            print(f"Verificación exitosa para {postor}: La puja revelada coincide con la puja cifrada.")
            pujas_reveladas_verificadas[postor] = puja_original_revelada_str
        else:
            print(f"Verificación fallida para {postor}: La puja revelada NO coincide con la puja cifrada.")
    else:
        print(f"Error: No se encontró puja cifrada para {postor}.")


=== Fase de Revelación y Verificación (con cifrado) ===
Verificación exitosa para postor_1: La puja revelada coincide con la puja cifrada.
Verificación exitosa para postor_2: La puja revelada coincide con la puja cifrada.
Verificación exitosa para postor_3: La puja revelada coincide con la puja cifrada.


## Determinar el ganador (con cifrado)

Alice compara las pujas reveladas y determina el ganador.


In [ ]:
# 1. Crear un diccionario para almacenar las pujas verificadas exitosamente
pujas_reveladas = pujas_reveladas_verificadas

# 2. Imprimir el diccionario pujas_reveladas
print("Pujas reveladas (verificadas):")
print(pujas_reveladas)

Pujas reveladas (verificadas):
{'postor_1': '150', 'postor_2': '250', 'postor_3': '200'}


## Flujo Completo del Protocolo de Compromiso (Método de Cifrado) - Ejecución

Esta celda ejecuta el flujo completo del protocolo de compromiso basado en cifrado, utilizando las funciones definidas anteriormente.

In [ ]:
# === Configuración de la Simulación ===
pujas_originales_cifrado_sim = {
    'postor_1': "150",
    'postor_2': "250",
    'postor_3': "200"
}

print("=== Inicio del Flujo del Protocolo de Cifrado ===")

# --- Paso 1: Generación de Claves Aleatorias y Compromiso (Cifrado) ---
print("\nPaso 1: Generación de claves y compromisos (cifrado) por los postores.")
claves_fernet_postores_cifrado_sim = {}
pujas_cifradas_postores_cifrado_sim = {}

for postor, puja in pujas_originales_cifrado_sim.items():
    clave_fernet = generar_clave_fernet() # Usamos la función definida anteriormente
    claves_fernet_postores_cifrado_sim[postor] = clave_fernet
    puja_cifrada = cifrar_puja(puja, clave_fernet) # Usamos la función definida anteriormente
    pujas_cifradas_postores_cifrado_sim[postor] = puja_cifrada
    print(f"  {postor}: Puja Original = {puja}, Compromiso (Puja Cifrada) = {puja_cifrada}")


# --- Paso 2: Envío de Compromisos a Alice ---
# Simular el envío de las pujas cifradas a Alice
pujas_cifradas_recibidas_por_alice_cifrado_sim = pujas_cifradas_postores_cifrado_sim

# === Comentario explicativo sobre el envío ===
print("\n--- Simulación de Envío ---")
print("Los postores envían sus pujas cifradas a Alice. En este punto, Alice tiene los compromisos pero no puede ver las pujas reales.")
print("Pujas cifradas recibidas por Alice:")
print(pujas_cifradas_recibidas_por_alice_cifrado_sim)
print("--------------------------")


# --- Paso 3: Fase de Revelación ---
# Los postores envían a Alice su puja original y la clave Fernet.
pujas_reveladas_info_cifrado_sim = {}
for postor in pujas_originales_cifrado_sim.keys():
     pujas_reveladas_info_cifrado_sim[postor] = {
         "puja_original": pujas_originales_cifrado_sim[postor],
         "clave_fernet": claves_fernet_postores_cifrado_sim[postor]
     }

print("\nPaso 3: Fase de Revelación - Postores revelan sus pujas y claves.")

# === Comentario explicativo sobre la revelación ===
print("\n--- Simulación de Envío ---")
print("Cada postor envía su puja original y la clave Fernet correspondiente a Alice.")
print("Información revelada recibida por Alice:")
print(pujas_reveladas_info_cifrado_sim)
print("--------------------------")


# --- Paso 4: Verificación por Alice ---
pujas_verificadas_cifrado_sim = {}

print("\nPaso 4: Verificación por Alice.")
for postor, info_revelada in pujas_reveladas_info_cifrado_sim.items():
    puja_original_revelada_str = info_revelada["puja_original"]
    clave_revelada = info_revelada["clave_fernet"]
    puja_cifrada_recibida = pujas_cifradas_recibidas_por_alice_cifrado_sim.get(postor)

    if puja_cifrada_recibida:
        verificacion_exitosa = descifrar_puja( # Usamos la función definida anteriormente
            puja_cifrada_recibida,
            clave_revelada,
            puja_original_revelada_str
        )

        if verificacion_exitosa:
            print(f"  Verificación exitosa para {postor}: La puja revelada coincide con la puja cifrada.")
            pujas_verificadas_cifrado_sim[postor] = puja_original_revelada_str
        else:
            print(f"  Verificación fallida para {postor}: La puja revelada NO coincide con la puja cifrada.")
    else:
        print(f"  Error: No se encontró puja cifrada para {postor}.")


# --- Paso 5: Determinación del Ganador ---
highest_bid_cifrado_sim = -1
winning_bidder_cifrado_sim = None

print("\nPaso 5: Determinación del Ganador.")
for postor, puja_str in pujas_verificadas_cifrado_sim.items():
    try:
        puja_int = int(puja_str)
        if puja_int > highest_bid_cifrado_sim:
            highest_bid_cifrado_sim = puja_int
            winning_bidder_cifrado_sim = postor
    except ValueError:
        print(f"  Advertencia: No se pudo convertir la puja '{puja_str}' de {postor} a un número entero. Saltando.")
        continue

if winning_bidder_cifrado_sim:
    print(f"\nEl ganador de la subasta (método de cifrado) es: {winning_bidder_cifrado_sim}")
    print(f"Con una puja de: {highest_bid_cifrado_sim}")
else:
    print("\nNo se encontraron pujas válidas para determinar un ganador (método de cifrado).")

print("\n=== Fin del Flujo del Protocolo de Cifrado ===")

=== Inicio del Flujo del Protocolo de Cifrado ===

Paso 1: Generación de claves y compromisos (cifrado) por los postores.
  postor_1: Puja Original = 150, Compromiso (Puja Cifrada) = b'gAAAAABpA8yuLt0AviX7J5F57ApjumKEhtd7J9EIq4Q57MfTM5xTxwEmqexG7NA0QPan1m8lFcH2QsnEP-jcxMbBwtIMjuutzw=='
  postor_2: Puja Original = 250, Compromiso (Puja Cifrada) = b'gAAAAABpA8yuwcJsEX17LGGKBADb9lp8uoQGYi0c6q-NiyKybGUAlzcjGbXYarOI18wBM0-5U9ULuDSubX34JOpdiy4mVOUqoQ=='
  postor_3: Puja Original = 200, Compromiso (Puja Cifrada) = b'gAAAAABpA8yuD1yjsv4q6kbrQOVFaxm-PdXVu5xKGlTyYEKU15Gm0z3PdrXL0wNYG3EOZDq-6V-2QiDKeFb2sB33sod4fPP1dA=='

--- Simulación de Envío ---
Los postores envían sus pujas cifradas a Alice. En este punto, Alice tiene los compromisos pero no puede ver las pujas reales.
Pujas cifradas recibidas por Alice:
{'postor_1': b'gAAAAABpA8yuLt0AviX7J5F57ApjumKEhtd7J9EIq4Q57MfTM5xTxwEmqexG7NA0QPan1m8lFcH2QsnEP-jcxMbBwtIMjuutzw==', 'postor_2': b'gAAAAABpA8yuwcJsEX17LGGKBADb9lp8uoQGYi0c6q-NiyKybGUAlzcjGbX

## Implementar el compromiso con hashing

En este método, cada postor genera un valor aleatorio secreto llamado "sal" y lo concatena con su puja. Luego, calcula el hash de esta combinación. Este hash es el compromiso que se envía a Alice.

In [ ]:
import hashlib
import os

def compromiso_con_hashing(puja, salt):
    """
    Calcula el hash SHA-256 de la puja concatenada con un salt.

    Parámetros:
    - puja: La puja como string.
    - salt: El salt como string.

    Retorna:
    - str: El hash SHA-256 en formato hexadecimal.
    """
    combinado = puja + salt
    hash_result = hashlib.sha256(combinado.encode()).hexdigest()
    return hash_result

def generar_salt_aleatorio():
    """
    Genera un salt aleatorio.

    Retorna:
    - str: El salt aleatorio en formato hexadecimal.
    """
    salt_bytes = os.urandom(16) # Generar 16 bytes aleatorios para el salt
    return salt_bytes.hex()

# === Simulación de postores generando compromiso con hashing ===
pujas_originales_hashing = {
    'postor_A': "300",
    'postor_B': "450",
    'postor_C': "350"
}

salts_postores = {}
compromisos_hashing_postores = {}

print("Generando compromisos con hashing para los postores:")
for postor, puja in pujas_originales_hashing.items():
    salt = generar_salt_aleatorio()
    salts_postores[postor] = salt
    compromiso = compromiso_con_hashing(puja, salt)
    compromisos_hashing_postores[postor] = compromiso
    print(f"  {postor}: Puja Original = {puja}, Salt = {salt}, Compromiso (Hash) = {compromiso}")

Generando compromisos con hashing para los postores:
  postor_A: Puja Original = 300, Salt = babfe1365c87643c8bcde0c6ed9a3b7d, Compromiso (Hash) = 3a2c2acd0850322c22d33a77b09036018f7f41ac6e7a129110f31801728e3925
  postor_B: Puja Original = 450, Salt = bb193a364372a80fbac76cf70e8e70b3, Compromiso (Hash) = 925849d90b139bc71ce0138474d071857202d627110baf89dbc276f66c220d82
  postor_C: Puja Original = 350, Salt = e15f67e7be916cbde441ef6496fd7d73, Compromiso (Hash) = c0c536a74c823a47c82683256aca5e7aea6edcb7ace820f232f3f26a6b68c6da


## Envío del compromiso (con hashing)

### Subtask:
Los postores envían sus hashes (compromisos) a Alice (la casa de subastas).

In [ ]:
# Simular la recepción de los compromisos (hashes) por parte de Alice
compromisos_hashing_recibidos_por_alice = compromisos_hashing_postores
# En un escenario real, Alice recibiría estos hashes de múltiples postores a través de la red

# Mostrar los compromisos (hashes) recibidos por Alice
print("\nCompromisos (hashes) recibidos por Alice:")
print(compromisos_hashing_recibidos_por_alice)


Compromisos (hashes) recibidos por Alice:
{'postor_A': '3a2c2acd0850322c22d33a77b09036018f7f41ac6e7a129110f31801728e3925', 'postor_B': '925849d90b139bc71ce0138474d071857202d627110baf89dbc276f66c220d82', 'postor_C': 'c0c536a74c823a47c82683256aca5e7aea6edcb7ace820f232f3f26a6b68c6da'}


## Fase de revelación (con hashing)

Cada postor revela su puja original y la sal que utilizó y se la envía a Alice. Alice puede entonces verificar el hash para comprobar que las pujas proporcionadas son compatibles con los compromisos que se hicieron en la primera fase.

In [ ]:
import hashlib

def verificar_compromiso_hashing(compromiso_original, puja_revelada, salt_revelado):
    """
    Verifica que el hash de la puja revelada y el salt coincida
    con el compromiso original recibido.

    Parámetros:
    - compromiso_original: El hash SHA-256 original recibido (string hexadecimal).
    - puja_revelada: La puja original revelada por el postor (string).
    - salt_revelado: El salt revelado por el postor (string hexadecimal).

    Retorna:
    - bool: True si los hashes coinciden, False en caso contrario.
    """
    combinado_revelado = puja_revelada + salt_revelado
    hash_calculado = hashlib.sha256(combinado_revelado.encode()).hexdigest()
    return hash_calculado == compromiso_original

# === Simular la fase de revelación para múltiples postores (con hashing) ===
pujas_reveladas_info_hashing = {
    'postor_A': {"puja_original": pujas_originales_hashing['postor_A'], "salt": salts_postores['postor_A']},
    'postor_B': {"puja_original": pujas_originales_hashing['postor_B'], "salt": salts_postores['postor_B']},
    'postor_C': {"puja_original": pujas_originales_hashing['postor_C'], "salt": salts_postores['postor_C']}
}

pujas_reveladas_verificadas_hashing = {}

print("\n=== Fase de Revelación y Verificación (con hashing) ===")
for postor, info_revelada in pujas_reveladas_info_hashing.items():
    puja_original_revelada_str = info_revelada["puja_original"]
    salt_revelado = info_revelada["salt"]
    compromiso_recibido = compromisos_hashing_recibidos_por_alice.get(postor)

    if compromiso_recibido:
        verificacion_exitosa = verificar_compromiso_hashing(
            compromiso_recibido,
            puja_original_revelada_str,
            salt_revelado
        )

        if verificacion_exitosa:
            print(f"Verificación exitosa para {postor}: La puja y salt revelados coinciden con el compromiso original.")
            pujas_reveladas_verificadas_hashing[postor] = puja_original_revelada_str
        else:
            print(f"Verificación fallida para {postor}: La puja y salt revelados NO coinciden con el compromiso original.")
    else:
        print(f"Error: No se encontró compromiso para {postor}.")


=== Fase de Revelación y Verificación (con hashing) ===
Verificación exitosa para postor_A: La puja y salt revelados coinciden con el compromiso original.
Verificación exitosa para postor_B: La puja y salt revelados coinciden con el compromiso original.
Verificación exitosa para postor_C: La puja y salt revelados coinciden con el compromiso original.


## Determinar el ganador (con hashing)

Alice compara las pujas reveladas y determina el ganador.

In [ ]:
# 1. Crear un diccionario para almacenar las pujas verificadas exitosamente para el método de hashing
pujas_reveladas_hashing = pujas_reveladas_verificadas_hashing

# 2. Imprimir el diccionario pujas_reveladas_hashing
print("\nPujas reveladas (verificadas con hashing):")
print(pujas_reveladas_hashing)


Pujas reveladas (verificadas con hashing):
{'postor_A': '300', 'postor_B': '450', 'postor_C': '350'}


# Cifrado Homomórfico

En este último bloque práctico vamos a ver un ejemplo muy sencillo de cifrado parcialmente homomórfico.

## Cifrado Homomórfico Parcial

El **cifrado homomórfico parcial** (PHE por sus siglas en inglés) es un tipo de cifrado que permite realizar ciertas operaciones matemáticas directamente sobre datos cifrados, sin necesidad de descifrarlos previamente. El resultado de la operación sobre los datos cifrados, al ser descifrado, es el mismo que se obtendría si la operación se hubiera realizado sobre los datos originales sin cifrar.

La característica clave del cifrado homomórfico parcial es que solo soporta un tipo específico de operación (por ejemplo, solo suma o solo multiplicación), o un número limitado de veces que se puede aplicar una operación. A diferencia del cifrado completamente homomórfico (FHE), que permite realizar un número arbitrario de operaciones sobre los datos cifrados, el PHE es más limitado en sus capacidades pero a menudo más eficiente y práctico de implementar.

**Relevancia:**

La capacidad de realizar cálculos sobre datos cifrados es fundamental para la privacidad de los datos en escenarios donde se procesan datos sensibles en entornos no confiables, como la computación en la nube. Permite a los proveedores de servicios en la nube realizar cálculos para sus clientes sin tener acceso a los datos originales, garantizando así la confidencialidad.

**RSA como Cifrado Parcialmente Homomórfico:**

El algoritmo de cifrado **RSA** es un ejemplo clásico de un criptosistema que es **parcialmente homomórfico bajo multiplicación**. Esto significa que si ciframos dos mensajes, `m1` y `m2`, como `C1 = Enc(m1)` y `C2 = Enc(m2)`, podemos multiplicar los textos cifrados (`C1 * C2`) y el resultado será el texto cifrado de la multiplicación de los mensajes originales (`Enc(m1 * m2)`). Matemáticamente, para RSA:

$$(m_1^e \pmod{n}) \times (m_2^e \pmod{n}) \equiv (m_1 \times m_2)^e \pmod{n}$$

Donde `e` es la clave pública de cifrado y `n` es el módulo. Esta propiedad de homomorfismo bajo multiplicación permite realizar ciertas operaciones multiplicativas directamente sobre los datos cifrados sin conocer la clave privada. Sin embargo, RSA no es homomórfico bajo suma, y solo soporta operaciones multiplicativas.


## Implementación de RSA simplificado (generación de claves)

Para generar las claves tenemos que buscar dos número primos, calcular n y phi, elegir un exponente publico e, y calcular el exponente privado d.


In [ ]:
import math

def generar_claves():
    """
    Genera un par de claves RSA simplificado (pública y privada) como diccionarios.

    Retorna:
    - dict: La clave pública {'n': n, 'e': e}.
    - dict: La clave privada {'n': n, 'd': d}.
    """
    # 1. Elegir dos números primos pequeños
    p = 11
    q = 13

    # 2. Calcular n
    n = p * q

    # 3. Calcular phi
    phi = (p - 1) * (q - 1)

    # 4. Elegir un exponente público e
    # Debe ser > 1, < phi, y gcd(e, phi) == 1
    e = 7 # Un valor común, gcd(7, 120) == 1

    # 5. Calcular el exponente privado d
    # Tal que (d * e) % phi == 1
    # Podemos usar un bucle simple para encontrar d
    d = 0
    while (d * e) % phi != 1:
        d += 1

    clave_publica = {'n': n, 'e': e}
    clave_privada = {'n': n, 'd': d}

    return clave_publica, clave_privada

# 8. Generar las claves
clave_publica, clave_privada = generar_claves()

# 9. Imprimir las claves
print(f"Clave Pública: {clave_publica}")
print(f"Clave Privada: {clave_privada}")

Clave Pública: {'n': 143, 'e': 7}
Clave Privada: {'n': 143, 'd': 103}


## Implementación de RSA simplificado (cifrado)

Se define una función para cifrar un mensaje `m` usando la clave pública  `(n, e)` con la formula `c = m^e % n`.


In [ ]:
def cifrar(m, clave_publica):
    """
    Cifra un mensaje m usando la clave pública RSA simplificada.

    Parámetros:
    - m: El mensaje a cifrar (entero).
    - clave_publica: La clave pública como diccionario {'n': n, 'e': e}.

    Retorna:
    - int: El texto cifrado c.
    """
    n = clave_publica['n']
    e = clave_publica['e']
    # Calcular el texto cifrado usando la fórmula c = m^e % n
    c = pow(m, e, n) # Utiliza la función pow(base, exp, mod) para cálculo eficiente
    return c

## Implementación de RSA simplificado (descifrado)

Ahora queremos descifrar un cifrado `c` usando la clave privada `(n, d)` con la formula `m = c^d % n`.


In [ ]:
def descifrar(c, clave_privada):
    """
    Descifra un texto cifrado c usando la clave privada RSA simplificada.

    Parámetros:
    - c: El texto cifrado a descifrar (entero).
    - clave_privada: La clave privada como diccionario {'n': n, 'd': d}.

    Retorna:
    - int: El mensaje descifrado m.
    """
    n = clave_privada['n']
    d = clave_privada['d']
    # Calcular el mensaje descifrado usando la fórmula m = c^d % n
    m = pow(c, d, n) # Utiliza la función pow(base, exp, mod) para cálculo eficiente
    return m

## Demostración de la propiedad homomórfica (multiplicación)

Ya estamos en disposición de poder comprobar como se cumple la propiedad del cifrado homomórfico multiplicativo

In [ ]:
# 1. Elegir dos mensajes enteros pequeños, m1 y m2 (menores que n)
m1 = 5
m2 = 7

# Asegurarse de que m1 y m2 son menores que n
n = clave_publica['n'] # Usar la clave global generada
if m1 >= n or m2 >= n:
    raise ValueError("Los mensajes deben ser menores que n")

# 2. Cifrar m1 usando la clave pública
c1 = cifrar(m1, clave_publica) # Usar la función cifrar y la clave global

# 3. Cifrar m2 usando la clave pública
c2 = cifrar(m2, clave_publica) # Usar la función cifrar y la clave global

# 4. Calcular el producto de los textos cifrados módulo n
c_producto = (c1 * c2) % clave_publica['n'] # Usar n de la clave pública

# 5. Descifrar c_producto usando la clave privada
m_producto = descifrar(c_producto, clave_privada) # Usar la función descifrar y la clave global

# 6. Calcular el producto de los mensajes originales
m_original_producto = m1 * m2

# 7. Imprimir los resultados
print(f"Mensaje 1 (m1): {m1}")
print(f"Mensaje 2 (m2): {m2}")
print(f"Clave Pública: {clave_publica}")
print(f"Clave Privada: {clave_privada}")
print(f"Texto cifrado de m1 (c1): {c1}")
print(f"Texto cifrado de m2 (c2): {c2}")
print(f"Producto de textos cifrados módulo n (c_producto): {c_producto}")
print(f"Descifrado de c_producto (m_producto): {m_producto}")
print(f"Producto de mensajes originales (m_original_producto): {m_original_producto}")

# 8. Comparar m_producto y m_original_producto y imprimir el resultado
if m_producto == m_original_producto:
    print("\nLa propiedad homomórfica parcial bajo multiplicación de RSA ha sido demostrada.")
else:
    print("\nLa propiedad homomórfica parcial bajo multiplicación de RSA NO ha sido demostrada.")

Mensaje 1 (m1): 5
Mensaje 2 (m2): 7
Clave Pública: {'n': 143, 'e': 7}
Clave Privada: {'n': 143, 'd': 103}
Texto cifrado de m1 (c1): 47
Texto cifrado de m2 (c2): 6
Producto de textos cifrados módulo n (c_producto): 139
Descifrado de c_producto (m_producto): 35
Producto de mensajes originales (m_original_producto): 35

La propiedad homomórfica parcial bajo multiplicación de RSA ha sido demostrada.


## Usando Claves RSA de 2048 bits

Para trabajar con tamaños de clave realistas como 2048 bits, utilizaremos la librería `cryptography` para la generación de claves, ya que la búsqueda de números primos grandes y la gestión de claves son tareas complejas. Sin embargo, seguiremos utilizando nuestras funciones simplificadas para el cifrado, descifrado y la demostración homomórfica, adaptadas para trabajar con los parámetros de las claves generadas por la librería.

### Generación de Claves RSA de 2048 bits (usando librería)

Generamos un par de claves RSA con un tamaño de 2048 bits utilizando la librería `cryptography`.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

def generar_claves_rsa_lib(key_size=2048):
    """
    Genera un par de claves RSA (pública y privada) usando la librería cryptography.

    Parámetros:
    - key_size: Tamaño de la clave en bits (por defecto 2048).

    Retorna:
    - clave_privada: La clave privada de RSA (objeto de la librería).
    - clave_publica: La clave pública de RSA (objeto de la librería).
    """
    clave_privada = rsa.generate_private_key(
        public_exponent=65537,
        key_size=key_size,
        backend=default_backend()
    )
    clave_publica = clave_privada.public_key()
    return clave_privada, clave_publica

# Generar las claves con 2048 bits
clave_privada_2048_lib, clave_publica_2048_lib = generar_claves_rsa_lib(key_size=2048)

# Para demostrar las operaciones manuales, necesitaremos obtener los parámetros (n, e, d)
# de las claves generadas por la librería.
n_2048_lib = clave_publica_2048_lib.public_numbers().n
e_2048_lib = clave_publica_2048_lib.public_numbers().e
d_2048_lib = clave_privada_2048_lib.private_numbers().d

print(f"Clave Pública (n, e) - 2048 bits: ({n_2048_lib}, {e_2048_lib})")
# No imprimimos d directamente ya que es la clave privada
print("Clave Privada (d) generada (no mostrada por seguridad)")

# Opcional: Imprimir claves en formato PEM (más común en la práctica)
# clave_publica_pem = clave_publica_2048_lib.public_bytes(
#     encoding=serialization.Encoding.PEM,
#     format=serialization.PublicFormat.SubjectPublicKeyInfo
# )
# clave_privada_pem = clave_privada_2048_lib.private_bytes(
#     encoding=serialization.Encoding.PEM,
#     format=serialization.PrivateFormat.PKCS8,
#     encryption_algorithm=serialization.NoEncryption()
# )
# print("\nClave Pública (PEM):")
# print(clave_publica_pem.decode())
# print("\nClave Privada (PEM):")
# print(clave_privada_pem.decode())

Clave Pública (n, e) - 2048 bits: (30127244325228386228507781607740539656530108704101682329443377524218659970965990878852041536713167510837000620591140162138452188927414531305555077377460418923075856504376611688251869005791879864458041402327883870984065064281579950310705016961843827920805883751502629974007808876334898292272572684147807992161340330386431910415370351609062770800618258761618577639326742110777839073328780078958379984209841106445996120009292814882069600101467568855521513507270371816803482854277008584517003881880734091771091113301354515411654099175014188165170904939667560509475586251567813973421605112313056356629768969223591770362501, 65537)
Clave Privada (d) generada (no mostrada por seguridad)


### Cifrado, Multiplicación Homomórfica y Descifrado con Claves de 2048 bits

Utilizamos nuestras funciones `cifrar` y `descifrar` manuales, pero con los parámetros (n, e, d) obtenidos de las claves de 2048 bits generadas por la librería para demostrar la propiedad homomórfica.

In [ ]:
# Asegúrate de que las funciones 'cifrar' y 'descifrar' definidas anteriormente
# en el notebook estén disponibles en este punto de la ejecución.

# 1. Elegir dos mensajes enteros pequeños, m1 y m2 (menores que n_2048_lib)
# Es importante que los mensajes sean menores que n_2048_lib para que el cifrado funcione correctamente
# en nuestro ejemplo simplificado. n_2048_lib es un número muy grande,
# por lo que la mayoría de los enteros pequeños funcionarán.
m1_grandes = 700
m2_grandes = 500

# Verificar que los mensajes son menores que n_2048_lib
if m1_grandes >= n_2048_lib or m2_grandes >= n_2048_lib:
     # En un escenario real, los mensajes serían acolchados (padded) antes de cifrar
     # para asegurar que m < n y añadir seguridad adicional.
     # Para esta demostración simplificada, simplemente lanzamos un error si m >= n.
     raise ValueError("Los mensajes deben ser menores que n para este ejemplo con claves grandes.")


print("=== Demostración de Cifrado Homomórfico Parcial (RSA - Multiplicación) con Claves de 2048 bits ===")
print(f"Clave Pública (n, e): ({n_2048_lib}, {e_2048_lib})")
# No mostramos la clave privada (d) directamente por seguridad.

print(f"Mensaje 1 (m1): {m1_grandes}")
print(f"Mensaje 2 (m2): {m2_grandes}")

# 2. Cifrar m1_grandes usando la función cifrar() con los parámetros de 2048 bits.
c1_grandes = cifrar(m1_grandes, {'n': n_2048_lib, 'e': e_2048_lib}) # Usar la función cifrar con el diccionario
print(f"Texto cifrado de m1 (c1): {c1_grandes}")

# 3. Cifrar m2_grandes usando la función cifrar() con los parámetros de 2048 bits.
c2_grandes = cifrar(m2_grandes, {'n': n_2048_lib, 'e': e_2048_lib}) # Usar la función cifrar con el diccionario
print(f"Texto cifrado de m2 (c2): {c2_grandes}")

# 4. Calcular el producto de los textos cifrados módulo n_2048_lib.
c_producto_grandes = (c1_grandes * c2_grandes) % n_2048_lib
print(f"Producto de textos cifrados módulo n (c_producto): {c_producto_grandes}")


# 5. Descifrar c_producto_grandes usando la función descifrar() con los parámetros de 2048 bits.
m_producto_grandes = descifrar(c_producto_grandes, {'n': n_2048_lib, 'd': d_2048_lib}) # Usar la función descifrar con el diccionario
print(f"Descifrado de c_producto (m_producto): {m_producto_grandes}")


# 6. Calcular el producto de los mensajes originales.
m_original_producto_grandes = m1_grandes * m2_grandes
print(f"Producto de mensajes originales (m_original_producto): {m_original_producto_grandes}")


# 7. Comparar m_producto_grandes y m_original_producto_grandes e imprimir un mensaje.
if m_producto_grandes == m_original_producto_grandes:
    print("\nResultado: ¡La propiedad homomórfica parcial bajo multiplicación de RSA ha sido demostrada con claves de 2048 bits!")
    print("El descifrado del producto de los cifrados es igual al producto de los mensajes originales.")
else:
    print("\nResultado: La propiedad homomórfica parcial bajo multiplicación de RSA NO ha sido demostrada con claves de 2048 bits.")
    print("El descifrado del producto de los cifrados NO es igual al producto de los mensajes originales.")

print("=== Fin de la Demostración con Claves Grandes ===")

=== Demostración de Cifrado Homomórfico Parcial (RSA - Multiplicación) con Claves de 2048 bits ===
Clave Pública (n, e): (30127244325228386228507781607740539656530108704101682329443377524218659970965990878852041536713167510837000620591140162138452188927414531305555077377460418923075856504376611688251869005791879864458041402327883870984065064281579950310705016961843827920805883751502629974007808876334898292272572684147807992161340330386431910415370351609062770800618258761618577639326742110777839073328780078958379984209841106445996120009292814882069600101467568855521513507270371816803482854277008584517003881880734091771091113301354515411654099175014188165170904939667560509475586251567813973421605112313056356629768969223591770362501, 65537)
Mensaje 1 (m1): 700
Mensaje 2 (m2): 500
Texto cifrado de m1 (c1): 449975612439663789633785471408281837520938816377690866647322734235526589379237299316233797693809045658031667670623350032027884995306910461556623113636724378746006628969095530451610879426